# Lab 08 · Best practices, and late data in streaming

**Part A** (about 55 min): measure what UDFs and loops cost, refactor and test a job, use broadcast variables and a star schema, and handle late events in a stream.<br>
**Part B** (about 60 min): 10 problems cleaning messy raw data: text, phone numbers, dates, numbers, JSON and arrays.

**Checklist questions Part A gives you evidence for:**<br>
<span style="color:#d93025"><b>6.1</b> Why avoid UDFs · <b>6.3</b> Broadcast variables · <b>6.4</b> The star schema benefit · <b>6.5</b> Why avoid lit() in loops · <b>6.6</b> Data lineage · <b>6.7</b> Spark SQL functions over Python logic · <b>6.8</b> Hardcoded values · <b>6.9</b> Late data in streaming</span>

## How to work
* Run the **Setup** cell first, then answer the questions in order. `F` is `pyspark.sql.functions`, `Window` is `pyspark.sql.Window`.
* **Part A**: run the code under the question (or add a line), look at the output, and write your answer as a comment.
* **Part B**: write the code yourself. Your result must look exactly like the **expected output**, with the rows in the same order.
* The answers are collapsed at the very end of the notebook. Open one only after you have tried.

## Documentation
* [Python UDFs](https://spark.apache.org/docs/latest/api/python/user_guide/udfandudtf.html) and [pandas UDFs](https://spark.apache.org/docs/latest/api/python/tutorial/sql/arrow_pandas.html#pandas-udfs-a-k-a-vectorized-udfs)
* [Broadcast variables](https://spark.apache.org/docs/latest/rdd-programming-guide.html#broadcast-variables)
* [Dynamic partition pruning setting](https://spark.apache.org/docs/latest/configuration.html#runtime-sql-configuration): `spark.sql.optimizer.dynamicPartitionPruning.enabled`
* [Testing PySpark](https://spark.apache.org/docs/latest/api/python/getting_started/testing_pyspark.html): `assertDataFrameEqual`
* [File metadata column](https://spark.apache.org/docs/latest/sql-data-sources-generic-options.html) (`_metadata.file_name`)
* [Structured Streaming: handling late data and watermarking](https://spark.apache.org/docs/latest/streaming/apis-on-dataframes-and-datasets.html#handling-late-data-and-watermarking)
* [Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html): `regexp_replace`, `initcap`, `try_to_date`, `from_json`, `filter`, `transform`, `aggregate`

## The data
Three small tables of an online shop. The setup cell creates them, and you can read every row in it.

* **customers**: *customer_id*, *name*, *email*, *city*, *signup_date* (the day the customer registered)
* **products**: *product_id*, *product_name*, *category*, *price* (dollars for one item)
* **orders**: *order_id*, *customer_id*, *product_id*, *quantity*, *order_date*, *status* (`delivered`, `shipped` or `cancelled`)

Good to know:
* order 1011 is a guest checkout: it has no *customer_id*
* customer 6 has no email and no city
* customer 7 has never ordered, and product 107 has never been sold
* order 1016 is for product 108, which is no longer in *products*

For this lab the setup cell also makes **signups_raw** (Part B): sign-ups exactly as a web form delivered them: *signup_id*, *full_name*, *email*, *phone*, *signup_date* (text, in three formats), *amount* (text) and *prefs* (JSON text). Everything is messy on purpose.

## Setup
Run this cell once. It starts Spark, saves the three shop tables as Parquet files in `data/shop/`, and creates *signups_raw* and the helpers `noop` and `best_time`.

In [ ]:
# Setup: starts Spark and creates the shop tables. Run it once.
import logging, os, shutil, sys, time
from datetime import date
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # Spark's Python workers use this notebook's Python
from pyspark.sql import SparkSession, Window, functions as F

spark = (SparkSession.builder
         .master("local[4]")   # 4 tasks at a time on your machine
         .appName("lab08")
         .config("spark.sql.adaptive.enabled", "true")        # AQE on, as in Spark's default
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "4")          # small data: 4 are enough
         .config("spark.ui.showConsoleProgress", "false")
         .config("spark.log.level", "FATAL")                  # hide Spark's log messages
         # Spark 4.2.0 bug (SPARK-60294): Python UDFs on Parquet can hang without this
         .config("spark.sql.execution.arrow.pythonUDF.columnarInput.enabled", "false")
         .getOrCreate())
for name in ("DataFrameQueryContextLogger", "SQLQueryContextLogger"):
    logging.getLogger(name).setLevel(logging.CRITICAL)        # errors still show, without an extra JSON log

customers_rows = [
    # customer_id, name, email, city, signup_date
    (1, "Aziz Karimov",     "aziz@mail.uz",     "Tashkent", date(2024, 1, 5)),
    (2, "Bella Schmidt",    "bella@web.de",     "Berlin",   date(2024, 1, 20)),
    (3, "Carlos Ruiz",      "carlos@correo.es", "Madrid",   date(2024, 2, 2)),
    (4, "Dilnoza Rahimova", "dilnoza@mail.uz",  "Tashkent", date(2024, 2, 14)),
    (5, "Emma Wilson",      "emma@post.co.uk",  "London",   date(2024, 3, 1)),
    (6, "Farid Aliyev",     None,               None,       date(2024, 3, 10)),
    (7, "Grace Lee",        "grace@web.de",     "Berlin",   date(2024, 3, 15)),
]
products_rows = [
    # product_id, product_name, category, price
    (101, "Laptop",       "Electronics", 1200.0),
    (102, "Headphones",   "Electronics",  150.0),
    (103, "Coffee Maker", "Kitchen",       85.0),
    (104, "Blender",      "Kitchen",       60.0),
    (105, "Novel",        "Books",         15.0),
    (106, "Cookbook",     "Books",         25.0),
    (107, "Desk Lamp",    "Home",          40.0),
]
orders_rows = [
    # order_id, customer_id, product_id, quantity, order_date, status
    (1001, 1,    101, 1, date(2024, 1, 10), "delivered"),
    (1002, 1,    105, 3, date(2024, 1, 10), "delivered"),
    (1003, 2,    102, 2, date(2024, 1, 25), "delivered"),
    (1004, 3,    103, 1, date(2024, 2, 3),  "cancelled"),
    (1005, 2,    106, 1, date(2024, 2, 5),  "delivered"),
    (1006, 4,    104, 2, date(2024, 2, 15), "shipped"),
    (1007, 1,    102, 1, date(2024, 2, 20), "delivered"),
    (1008, 5,    101, 1, date(2024, 3, 2),  "shipped"),
    (1009, 3,    105, 4, date(2024, 3, 5),  "delivered"),
    (1010, 4,    106, 2, date(2024, 3, 8),  "cancelled"),
    (1011, None, 104, 1, date(2024, 3, 9),  "delivered"),
    (1012, 6,    103, 2, date(2024, 3, 12), "shipped"),
    (1013, 2,    101, 1, date(2024, 3, 18), "delivered"),
    (1014, 5,    105, 5, date(2024, 3, 20), "delivered"),
    (1015, 1,    103, 1, date(2024, 3, 25), "shipped"),
    (1016, 3,    108, 1, date(2024, 3, 28), "delivered"),
]

# Write the tables as Parquet files and read them back, the way real data arrives.
# data/shop belongs to this lab: it is rebuilt every time you run this cell.
shutil.rmtree("data/shop", ignore_errors=True)
for table, rows, schema in [
    ("customers", customers_rows, "customer_id INT, name STRING, email STRING, city STRING, signup_date DATE"),
    ("products",  products_rows,  "product_id INT, product_name STRING, category STRING, price DOUBLE"),
    ("orders",    orders_rows,    "order_id INT, customer_id INT, product_id INT, quantity INT, order_date DATE, status STRING"),
]:
    spark.createDataFrame(rows, schema).coalesce(1).write.parquet(f"data/shop/{table}")

customers = spark.read.parquet("data/shop/customers")
products = spark.read.parquet("data/shop/products")
orders = spark.read.parquet("data/shop/orders")

def noop(df):
    """Runs the whole query but keeps no result (Spark's "noop" sink)."""
    df.write.format("noop").mode("overwrite").save()

def best_time(action, runs=3):
    """Runs action() a few times and returns the fastest time in seconds (the first run warms up)."""
    times = []
    for _ in range(runs):
        start = time.time()
        action()
        times.append(time.time() - start)
    return round(min(times), 2)

# Part B: raw sign-ups, exactly as a web form delivered them
signups_raw = spark.createDataFrame([
    # signup_id, full_name, email, phone, signup_date (text), amount (text), prefs (JSON text)
    (1, "  aziz   KARIMOV ", "Aziz@Mail.UZ ",      "+998 90 123-45-67", "2024-01-05", "12.50", '{"newsletter": true, "lang": "uz"}'),
    (2, "bella schmidt",     "bella@web.de",       "0049 151 2345678",  "20/01/2024", "abc",   '{"newsletter": false, "lang": "de"}'),
    (3, "Carlos  Ruiz",      "",                   "(34) 612 345 678",  "2024/02/02", "7",     '{"lang": "es"}'),
    (4, "DILNOZA rahimova",  "dilnoza@mail.uz",    "90-765-43-21",      "14/02/2024", "",      None),
    (5, "Emma Wilson",       "emma[at]post.co.uk", "07700 900123",      "2024-03-01", "3.75",  '{"newsletter": true}'),
    (6, "aziz karimov",      "aziz@mail.uz",       "+998901234567",     "2024-03-02", "1.00",  '{"newsletter": false, "lang": "uz"}'),
], "signup_id INT, full_name STRING, email STRING, phone STRING, signup_date STRING, amount STRING, prefs STRING")

print("Spark", spark.version, "| Spark UI:", spark.sparkContext.uiWebUrl)

Look at the tables:

In [ ]:
customers.show()
products.show()
orders.show()
signups_raw.show(truncate=False)

## Part A · Habits of a good Spark job

### UDFs or built-in functions <span style="color:#d93025">(6.1, 6.7)</span>

**A1. Add 12% VAT to 10 million prices in three ways: a built-in expression, a Python UDF and a pandas UDF. Which is the slowest?**

In [ ]:
# My answer:
import pandas as pd
prices = spark.range(10_000_000).select((F.col("id") % 1000 + 0.99).alias("price"))
vat_udf = F.udf(lambda p: round(p * 1.12, 2), "double", useArrow=False)   # a plain Python UDF: one row at a time

@F.pandas_udf("double")
def vat_pandas(p: pd.Series) -> pd.Series:                                 # a pandas UDF: a batch of rows at a time
    return (p * 1.12).round(2)

for name, column in [("built-in", F.round(F.col("price") * 1.12, 2)), ("python udf", vat_udf("price")),
                     ("pandas udf", vat_pandas("price"))]:
    print(f"{name:10}", best_time(lambda: noop(prices.select(column.alias("with_vat")))), "s")

**A2. Filter a Parquet copy of the prices with `price > 900`, once as a built-in condition and once with the Python UDF `over_900`. Which filter is pushed down to the file scan?**

In [ ]:
# My answer:
prices.write.mode("overwrite").parquet("data/shop/prices")
stored = spark.read.parquet("data/shop/prices")
over_900 = F.udf(lambda p: p > 900, "boolean")
stored.filter(F.col("price") > 900).explain()

### Clean code <span style="color:#d93025">(6.5, 6.8)</span>

**A3. Add 300 constant columns, once with `withColumn(..., F.lit(i))` in a loop and once with a single `select`. How long does building each DataFrame take (nothing runs yet)?**

In [ ]:
# My answer:
start = time.time()
looped = spark.range(10)
for i in range(300):
    looped = looped.withColumn(f"c{i}", F.lit(i))
print(f"loop:   {time.time() - start:.2f} s")

**A4. `bad_report` hardcodes its status, quantity and VAT rate. Write `good_report(orders, products, config)` that takes them from `config`. Do both return the same table?**

In [ ]:
# My answer:
def bad_report(orders, products):
    """Revenue with 12% VAT per category of the delivered orders with 2 or more items. Every value is hardcoded."""
    big = orders.filter((F.col("status") == "delivered") & (F.col("quantity") >= 2))
    return (big.join(products, "product_id")
               .withColumn("revenue", F.round(F.col("quantity") * F.col("price") * 1.12, 2))
               .groupBy("category").agg(F.sum("revenue").alias("revenue"))
               .orderBy("category"))

config = {"status": "delivered", "min_quantity": 2, "vat_rate": 0.12}
bad_report(orders, products).show()

**A5. Test `good_report` with tiny DataFrames you build yourself (cell below) and `assertDataFrameEqual`. Which revenue should come out? Change your expected number once to see a failing test.**

In [ ]:
# My answer:
from pyspark.testing import assertDataFrameEqual
test_orders = spark.createDataFrame(
    [(1, 101, 2, "delivered"), (2, 101, 1, "delivered"), (3, 102, 5, "cancelled")],
    "order_id INT, product_id INT, quantity INT, status STRING")
test_products = spark.createDataFrame([(101, "A", 10.0), (102, "B", 1.0)], "product_id INT, category STRING, price DOUBLE")

### Broadcast variables and star schemas <span style="color:#d93025">(6.3, 6.4)</span>

**A6. Send the category margins to the executors once with a broadcast variable and use it in a UDF (cell below). How can you get the same result without a UDF?**

In [ ]:
# My answer:
margins = {"Electronics": 0.10, "Kitchen": 0.25, "Books": 0.30, "Home": 0.20}
margins_bc = spark.sparkContext.broadcast(margins)          # sent once to every executor
margin_of = F.udf(lambda category: margins_bc.value.get(category), "double")
products.withColumn("margin", margin_of("category")).show()

**A7. Star schema: join a fact table partitioned by date with a small date dimension, filtered to Saturdays. How many of the fact table's files does Spark read?**

In [ ]:
# My answer:
from datetime import timedelta
fact = spark.range(1_000_000).select(
    F.col("id").alias("order_id"),
    F.date_add(F.lit("2024-01-01").cast("date"), F.pmod(F.xxhash64("id"), F.lit(91)).cast("int")).alias("order_date"),
    F.pmod(F.xxhash64("id", F.lit(2)), F.lit(1000)).cast("double").alias("amount"))
fact.write.mode("overwrite").partitionBy("order_date").parquet("data/shop/fact")       # a big fact table, one folder per day
fact = spark.read.parquet("data/shop/fact")
days = [date(2024, 1, 1) + timedelta(days=i) for i in range(91)]
dates = spark.createDataFrame([(d, d.strftime("%a")) for d in days], "order_date DATE, weekday STRING")   # a small date dimension
saturday_sales = fact.join(dates.filter(F.col("weekday") == "Sat"), "order_date").agg(F.sum("amount"))

### Data lineage <span style="color:#d93025">(6.6)</span>

**A8. Where did each row come from? Add the source file to the prices with the hidden `_metadata` column and count the rows per file.**

In [ ]:
# My answer:
(spark.read.parquet("data/shop/prices")
      .withColumn("source_file", F.col("_metadata.file_name"))
      .groupBy("source_file").count()
      .show(truncate=False))

### Late data in streaming <span style="color:#d93025">(6.9)</span>

**A9. Events arrive in three batches; the stream counts them per 5-minute window with a 10-minute watermark. When is a window written, and what happens to the late event at 10:03 in batch 3?**

In [ ]:
# My answer:
import json
STREAM_IN, STREAM_OUT, CHECKPOINT = "data/shop/stream_in", "data/shop/stream_out", "data/shop/stream_checkpoint"
for folder in [STREAM_IN, STREAM_OUT, CHECKPOINT]:
    shutil.rmtree(folder, ignore_errors=True)
os.makedirs(STREAM_IN)

def new_events(batch, times):
    """Drops one JSON file of events (their times on 2024-03-01) into the input folder."""
    with open(f"{STREAM_IN}/{batch}.json", "w") as f:
        for t in times:
            f.write(json.dumps({"event_time": f"2024-03-01 {t}:00"}) + "\n")

def run_stream():
    """Processes the new files, then prints every window written so far."""
    (spark.readStream.schema("event_time TIMESTAMP").json(STREAM_IN)
          .withWatermark("event_time", "10 minutes")                   # events may be up to 10 minutes late
          .groupBy(F.window("event_time", "5 minutes")).count()       # count per 5-minute window
          .writeStream.outputMode("append").format("parquet")
          .option("path", STREAM_OUT).option("checkpointLocation", CHECKPOINT)
          .trigger(availableNow=True).start().awaitTermination())
    (spark.read.parquet(STREAM_OUT)
          .select(F.date_format("window.start", "HH:mm").alias("window_start"), "count")
          .orderBy("window_start").show())

new_events("batch1", ["10:00", "10:02", "10:07"]); run_stream()
new_events("batch2", ["10:25"]);                   run_stream()
new_events("batch3", ["10:03", "10:31"]);          run_stream()

**Done with Part A?** Explain each red checklist question at the top aloud, in your own words, using what you saw.

## Part B · Cleaning messy data
10 problems, mostly on *signups_raw*. Use built-in functions only (no UDFs). Where an expected output is left-aligned, print with `.show(truncate=False)`.

**B1. Clean the names: no spaces at the ends, one space between words, each word capitalized (`Aziz Karimov`). Show signup_id and name.**

Expected output:

```
+---------+----------------+
|signup_id|            name|
+---------+----------------+
|        1|    Aziz Karimov|
|        2|   Bella Schmidt|
|        3|     Carlos Ruiz|
|        4|Dilnoza Rahimova|
|        5|     Emma Wilson|
|        6|    Aziz Karimov|
+---------+----------------+
```

<details><summary>Hint</summary>

`F.trim`, then `F.regexp_replace(column, r"\s+", " ")` for repeated spaces, then `F.initcap`.

</details>

**B2. Clean the emails: no spaces at the ends and lower case; an empty email becomes null. Show signup_id and email.**

Expected output:

```
+---------+------------------+
|signup_id|email             |
+---------+------------------+
|1        |aziz@mail.uz      |
|2        |bella@web.de      |
|3        |NULL              |
|4        |dilnoza@mail.uz   |
|5        |emma[at]post.co.uk|
|6        |aziz@mail.uz      |
+---------+------------------+
```

<details><summary>Hint</summary>

`F.when(F.trim("email") == "", None).otherwise(...)`.

</details>

**B3. Flag the valid emails (after cleaning as in B2): text, one `@`, text, a dot, and letters at the end. A missing email is not valid. Show signup_id, email and valid (true or false).**

Expected output:

```
+---------+------------------+-----+
|signup_id|email             |valid|
+---------+------------------+-----+
|1        |aziz@mail.uz      |true |
|2        |bella@web.de      |true |
|3        |NULL              |false|
|4        |dilnoza@mail.uz   |true |
|5        |emma[at]post.co.uk|false|
|6        |aziz@mail.uz      |true |
+---------+------------------+-----+
```

<details><summary>Hint</summary>

`Column.rlike(pattern)` tests a regular expression; on null it gives null, so `F.coalesce(..., F.lit(False))`.

</details>

**B4. Keep only the digits of each phone number (`digits`), and its last 9 digits (`phone9`). Show signup_id, digits and phone9.**

Expected output:

```
+---------+--------------+---------+
|signup_id|        digits|   phone9|
+---------+--------------+---------+
|        1|  998901234567|901234567|
|        2|00491512345678|512345678|
|        3|   34612345678|612345678|
|        4|     907654321|907654321|
|        5|   07700900123|700900123|
|        6|  998901234567|901234567|
+---------+--------------+---------+
```

<details><summary>Hint</summary>

`F.regexp_replace(column, "[^0-9]", "")` removes everything that is not a digit; `F.substring(column, -9, 9)` takes the last 9 characters.

</details>

**B5. The sign-up dates come in three formats (`2024-01-05`, `20/01/2024`, `2024/02/02`). Turn them into one date column `signup_day`. Show signup_id, signup_date and signup_day.**

Expected output:

```
+---------+-----------+----------+
|signup_id|signup_date|signup_day|
+---------+-----------+----------+
|        1| 2024-01-05|2024-01-05|
|        2| 20/01/2024|2024-01-20|
|        3| 2024/02/02|2024-02-02|
|        4| 14/02/2024|2024-02-14|
|        5| 2024-03-01|2024-03-01|
|        6| 2024-03-02|2024-03-02|
+---------+-----------+----------+
```

<details><summary>Hint</summary>

`F.try_to_date(column, pattern)` returns null when the text doesn't match (plain `to_date` raises an error); `F.coalesce` takes the first one that worked.

</details>

**B6. Turn amount (text) into a number `amount_value`; text that is not a number becomes null. Show signup_id, amount and amount_value.**

Expected output:

```
+---------+------+------------+
|signup_id|amount|amount_value|
+---------+------+------------+
|        1| 12.50|        12.5|
|        2|   abc|        NULL|
|        3|     7|         7.0|
|        4|      |        NULL|
|        5|  3.75|        3.75|
|        6|  1.00|         1.0|
+---------+------+------------+
```

<details><summary>Hint</summary>

`Column.try_cast("double")` gives null instead of an error.

</details>

**B7. Read `newsletter` and `lang` from the JSON text in prefs (missing values become null). Show signup_id, newsletter and lang.**

Expected output:

```
+---------+----------+----+
|signup_id|newsletter|lang|
+---------+----------+----+
|        1|      true|  uz|
|        2|     false|  de|
|        3|      NULL|  es|
|        4|      NULL|NULL|
|        5|      true|NULL|
|        6|     false|  uz|
+---------+----------+----+
```

<details><summary>Hint</summary>

`F.from_json(column, "newsletter BOOLEAN, lang STRING")` turns the text into a struct; take its fields with `["newsletter"]`.

</details>

**B8. The same person signed up twice with differently written emails. Clean the emails (as in B2) and keep only the latest sign-up (largest signup_id) of each email; sign-ups without an email stay. Show signup_id and email.**

Expected output:

```
+---------+------------------+
|signup_id|email             |
+---------+------------------+
|2        |bella@web.de      |
|3        |NULL              |
|4        |dilnoza@mail.uz   |
|5        |emma[at]post.co.uk|
|6        |aziz@mail.uz      |
+---------+------------------+
```

<details><summary>Hint</summary>

Clean first, then `row_number` per email (newest first) and keep number 1.

</details>

**B9. For each customer, collect the quantities of their orders into a sorted array, then keep only the quantities above 1 (`above_1`) and add those up (`total_above_1`), without `explode`. Leave out the guest order.**

Expected output:

```
+-----------+------------+-------+-------------+
|customer_id|  quantities|above_1|total_above_1|
+-----------+------------+-------+-------------+
|          1|[1, 1, 1, 3]|    [3]|            3|
|          2|   [1, 1, 2]|    [2]|            2|
|          3|   [1, 1, 4]|    [4]|            4|
|          4|      [2, 2]| [2, 2]|            4|
|          5|      [1, 5]|    [5]|            5|
|          6|         [2]|    [2]|            2|
+-----------+------------+-------+-------------+
```

<details><summary>Hint</summary>

Higher-order functions work inside arrays: `F.filter(array, lambda x: ...)` and `F.aggregate(array, start, lambda total, x: ...)`.

</details>

**B10. For each category, collect the product prices into a sorted array, and add `with_vat`: every price × 1.12, rounded to 2 decimals. Show category, prices and with_vat.**

Expected output:

```
+-----------+---------------+---------------+
|category   |prices         |with_vat       |
+-----------+---------------+---------------+
|Books      |[15.0, 25.0]   |[16.8, 28.0]   |
|Electronics|[150.0, 1200.0]|[168.0, 1344.0]|
|Home       |[40.0]         |[44.8]         |
|Kitchen    |[60.0, 85.0]   |[67.2, 95.2]   |
+-----------+---------------+---------------+
```

<details><summary>Hint</summary>

`F.transform(array, lambda x: ...)` changes every element.

</details>

---
**Done?** In the Theory part of Day 8, answer the red checklist questions aloud.

## Answers
Try each question yourself first. Click a question to open its answer; Part A answers also show the output of the reference run.

<details>
<summary><b>A1</b> &nbsp;Add 12% VAT to 10 million prices in three ways: a built-in expression, a Python UDF and a pandas UDF. Which ...</summary>

**The plain Python UDF, by far** (the built-in expression and the pandas UDF are several times faster). A Python UDF sends every row from the JVM to a Python worker, runs your function one row at a time and sends the result back: the copying and Python's speed cost you. A pandas UDF moves the data in Arrow batches and computes with vectorized pandas, which is much faster; Spark 4 also uses Arrow for plain UDFs by default (`useArrow=False` shows the old row-by-row way). Built-in functions run inside the JVM as generated code, and Catalyst understands them (A2).

```python
import pandas as pd
prices = spark.range(10_000_000).select((F.col("id") % 1000 + 0.99).alias("price"))
vat_udf = F.udf(lambda p: round(p * 1.12, 2), "double", useArrow=False)   # a plain Python UDF: one row at a time

@F.pandas_udf("double")
def vat_pandas(p: pd.Series) -> pd.Series:                                 # a pandas UDF: a batch of rows at a time
    return (p * 1.12).round(2)

for name, column in [("built-in", F.round(F.col("price") * 1.12, 2)), ("python udf", vat_udf("price")),
                     ("pandas udf", vat_pandas("price"))]:
    print(f"{name:10}", best_time(lambda: noop(prices.select(column.alias("with_vat")))), "s")
```

Result:

```
built-in   0.36 s
python udf 1.81 s
pandas udf 0.46 s
```

*Your times will differ.*

</details>

<details>
<summary><b>A2</b> &nbsp;Filter a Parquet copy of the prices with price &gt; 900, once as a built-in condition and once with the Python ...</summary>

**Only the built-in one**: `PushedFilters: [IsNotNull(price), GreaterThan(price,900.0)]`, against `PushedFilters: []` with the UDF. To Catalyst a UDF is a black box: it can't push it into the scan, reorder it, fold it or use statistics, so Spark must read every row first. That's why you prefer the standard Spark SQL functions in `pyspark.sql.functions`: they are faster *and* the optimizer can work with them. Write a UDF only when no built-in function can do the job, and then prefer a pandas UDF.

```python
prices.write.mode("overwrite").parquet("data/shop/prices")
stored = spark.read.parquet("data/shop/prices")
over_900 = F.udf(lambda p: p > 900, "boolean")
stored.filter(F.col("price") > 900).explain()
stored.filter(over_900("price")).explain()
```

Result:

```
   +- FileScan parquet [price#129] Batched: true, DataFilters: [isnotnull(price#129), (price#129 > 900.0)], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [IsNotNull(price), GreaterThan(price,900.0)], ReadSchema: struct<price:double>
         +- FileScan parquet [price#129] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<price:double>
```

</details>

<details>
<summary><b>A3</b> &nbsp;Add 300 constant columns, once with withColumn(..., F.lit(i)) in a loop and once with a single select. How ...</summary>

**The loop is about 10 times slower, before anything runs.** Each `withColumn` creates a new DataFrame with one more projection, and Spark analyzes the whole growing plan again each time: 300 steps make a huge plan and slow analysis (thousands can even fail with a stack overflow). Build all the columns in one `select("*", *[...])`, or with `withColumns({...})`.

```python
start = time.time()
looped = spark.range(10)
for i in range(300):
    looped = looped.withColumn(f"c{i}", F.lit(i))
print(f"loop:   {time.time() - start:.2f} s")
start = time.time()
selected = spark.range(10).select("*", *[F.lit(i).alias(f"c{i}") for i in range(300)])
print(f"select: {time.time() - start:.2f} s")
```

Result:

```
loop:   2.18 s
select: 0.19 s
```

*Your times will differ.*

</details>

<details>
<summary><b>A4</b> &nbsp;bad_report hardcodes its status, quantity and VAT rate. Write good_report(orders, products, config) that ...</summary>

**Yes, the same table.** Pass settings in instead of writing them into the code: a `config` dict (filled from a JSON or YAML file, job arguments or environment variables) that you load once and hand to your functions. Then one job works for every environment (dev, test, prod), a change of rate doesn't need a code change, and tests can pass their own values (A5).

```python
def bad_report(orders, products):
    """Revenue with 12% VAT per category of the delivered orders with 2 or more items. Every value is hardcoded."""
    big = orders.filter((F.col("status") == "delivered") & (F.col("quantity") >= 2))
    return (big.join(products, "product_id")
               .withColumn("revenue", F.round(F.col("quantity") * F.col("price") * 1.12, 2))
               .groupBy("category").agg(F.sum("revenue").alias("revenue"))
               .orderBy("category"))

config = {"status": "delivered", "min_quantity": 2, "vat_rate": 0.12}
bad_report(orders, products).show()

def good_report(orders, products, config):
    """The same report; every value comes from config."""
    big = orders.filter((F.col("status") == config["status"]) & (F.col("quantity") >= config["min_quantity"]))
    return (big.join(products, "product_id")
               .withColumn("revenue", F.round(F.col("quantity") * F.col("price") * (1 + config["vat_rate"]), 2))
               .groupBy("category").agg(F.sum("revenue").alias("revenue"))
               .orderBy("category"))

good_report(orders, products, config).show()
```

Result:

```
+-----------+-------+
|   category|revenue|
+-----------+-------+
|      Books|  201.6|
|Electronics|  336.0|
+-----------+-------+

+-----------+-------+
|   category|revenue|
+-----------+-------+
|      Books|  201.6|
|Electronics|  336.0|
+-----------+-------+
```

</details>

<details>
<summary><b>A5</b> &nbsp;Test good_report with tiny DataFrames you build yourself (cell below) and assertDataFrameEqual. Which revenue ...</summary>

**Category A with revenue 22.4**: only order 1 is delivered with 2 or more items (2 × 10 × 1.12). `assertDataFrameEqual(actual, expected)` compares schemas and rows and prints the rows that differ. Real projects keep such tests in `pytest` files and run them on every change: small hand-made DataFrames that cover the edge cases (a cancelled order, quantity 1, a missing product).

```python
from pyspark.testing import assertDataFrameEqual
test_orders = spark.createDataFrame(
    [(1, 101, 2, "delivered"), (2, 101, 1, "delivered"), (3, 102, 5, "cancelled")],
    "order_id INT, product_id INT, quantity INT, status STRING")
test_products = spark.createDataFrame([(101, "A", 10.0), (102, "B", 1.0)], "product_id INT, category STRING, price DOUBLE")
expected = spark.createDataFrame([("A", 22.4)], "category STRING, revenue DOUBLE")
assertDataFrameEqual(good_report(test_orders, test_products, config), expected)
print("test passed")
```

Result:

```
test passed
```

</details>

<details>
<summary><b>A6</b> &nbsp;Send the category margins to the executors once with a broadcast variable and use it in a UDF (cell below). ...</summary>

**With a broadcast join of a small DataFrame.** A **broadcast variable** ships a read-only value (a dict, a model, a set of ids) to each executor once and keeps it there; without it, the value would be copied into every task. But with DataFrames, turn the lookup into a small DataFrame and join it with `F.broadcast(...)`: same effect, no Python UDF, and Catalyst can optimize it.

```python
margins = {"Electronics": 0.10, "Kitchen": 0.25, "Books": 0.30, "Home": 0.20}
margins_bc = spark.sparkContext.broadcast(margins)          # sent once to every executor
margin_of = F.udf(lambda category: margins_bc.value.get(category), "double")
products.withColumn("margin", margin_of("category")).show()
margins_df = spark.createDataFrame(list(margins.items()), "category STRING, margin DOUBLE")
products.join(F.broadcast(margins_df), "category").orderBy("product_id").show()
```

Result:

```
+----------+------------+-----------+------+------+
|product_id|product_name|   category| price|margin|
+----------+------------+-----------+------+------+
|       101|      Laptop|Electronics|1200.0|   0.1|
|       102|  Headphones|Electronics| 150.0|   0.1|
|       103|Coffee Maker|    Kitchen|  85.0|  0.25|
|       104|     Blender|    Kitchen|  60.0|  0.25|
|       105|       Novel|      Books|  15.0|   0.3|
|       106|    Cookbook|      Books|  25.0|   0.3|
|       107|   Desk Lamp|       Home|  40.0|   0.2|
+----------+------------+-----------+------+------+

+-----------+----------+------------+------+------+
|   category|product_id|product_name| price|margin|
+-----------+----------+------------+------+------+
|Electronics|       101|      Laptop|1200.0|   0.1|
|Electronics|       102|  Headphones| 150.0|   0.1|
|    Kitchen|       103|Coffee Maker|  85.0|  0.25|
|    Kitchen|       104|     Blender|  60.0|  0.25|
|      Books|       105|       Novel|  15.0|   0.3|
|      Books|       106|    Cookbook|  25.0|   0.3|
|       Home|       107|   Desk Lamp|  40.0|   0.2|
+-----------+----------+------------+------+------+
```

</details>

<details>
<summary><b>A7</b> &nbsp;Star schema: join a fact table partitioned by date with a small date dimension, filtered to Saturdays. How ...</summary>

**52 of 364 files** (13 Saturdays × 4 files). You filtered the small dimension, not the fact table, yet Spark skipped the other days' folders: **dynamic partition pruning** (`dynamicpruningexpression` in the scan's PartitionFilters). Spark first runs the dimension side, collects the matching dates, and uses them to prune the fact table's partitions. That is the benefit of a **star schema** in Spark: one big fact table joined with small dimension tables, which are broadcast and whose filters prune the fact table.

```python
from datetime import timedelta
fact = spark.range(1_000_000).select(
    F.col("id").alias("order_id"),
    F.date_add(F.lit("2024-01-01").cast("date"), F.pmod(F.xxhash64("id"), F.lit(91)).cast("int")).alias("order_date"),
    F.pmod(F.xxhash64("id", F.lit(2)), F.lit(1000)).cast("double").alias("amount"))
fact.write.mode("overwrite").partitionBy("order_date").parquet("data/shop/fact")       # a big fact table, one folder per day
fact = spark.read.parquet("data/shop/fact")
days = [date(2024, 1, 1) + timedelta(days=i) for i in range(91)]
dates = spark.createDataFrame([(d, d.strftime("%a")) for d in days], "order_date DATE, weekday STRING")   # a small date dimension
saturday_sales = fact.join(dates.filter(F.col("weekday") == "Sat"), "order_date").agg(F.sum("amount"))
saturday_sales.collect()
saturday_sales.explain()
saturdays = dates.filter(F.col("weekday") == "Sat")
read = fact.select("*", "_metadata.file_path").join(saturdays, "order_date").select("file_path").distinct().count()
print(read, "of", fact.select("_metadata.file_path").distinct().count(), "files read")
```

Result:

```
                     :        +- FileScan parquet [amount#856,order_date#857] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [isnotnull(order_date#857), dynamicpruningexpression(true)], PushedFilters: [], ReadSchema: struct<amount:double>
               :  +- FileScan parquet [amount#856,order_date#857] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/..., PartitionFilters: [isnotnull(order_date#857), dynamicpruningexpression(order_date#857 IN dynamicpruning#867)], PushedFilters: [], ReadSchema: struct<amount:double>
52 of 364 files read
```

</details>

<details>
<summary><b>A8</b> &nbsp;Where did each row come from? Add the source file to the prices with the hidden _metadata column and count ...</summary>

**4 files with 2,500,000 rows each.** Every file-based DataFrame has a hidden `_metadata` column (`file_path`, `file_name`, `file_size`, `file_modification_time`); keeping it as a `source_file` column, for example in a bronze table (Lab 10), records where each row came from. Inside a job, Spark's own lineage is the plan (`explain()`, the DAG in the Spark UI). Across jobs and tables, teams use lineage tools (OpenLineage, a catalog such as Unity Catalog) and Delta's table history (Lab 09).

```python
(spark.read.parquet("data/shop/prices")
      .withColumn("source_file", F.col("_metadata.file_name"))
      .groupBy("source_file").count()
      .show(truncate=False))
```

Result:

```
+-------------------------------------------------------------------+-------+
|source_file                                                        |count  |
+-------------------------------------------------------------------+-------+
|part-00003-c30c094f-6c3f-495d-b5de-b197b34b2e8b-c000.snappy.parquet|2500000|
|part-00002-c30c094f-6c3f-495d-b5de-b197b34b2e8b-c000.snappy.parquet|2500000|
|part-00000-c30c094f-6c3f-495d-b5de-b197b34b2e8b-c000.snappy.parquet|2500000|
|part-00001-c30c094f-6c3f-495d-b5de-b197b34b2e8b-c000.snappy.parquet|2500000|
+-------------------------------------------------------------------+-------+
```

</details>

<details>
<summary><b>A9</b> &nbsp;Events arrive in three batches; the stream counts them per 5-minute window with a 10-minute watermark. When ...</summary>

**A window is written only when the watermark passes its end, and the 10:03 event is dropped.** The **watermark** is the latest event time seen minus 10 minutes. After batch 1 it is 09:57, so no window is final yet and nothing is written. After batch 2 (10:25) it is 10:15, so the 10:00 and 10:05 windows are complete and written once (append mode). In batch 3 the 10:03 event is older than the watermark, so Spark ignores it and the 10:00 window keeps its count of 2. That is how you **handle late data**: you decide how late an event may be; events within that delay still update their window, later ones are dropped, and Spark can forget the state of old windows instead of keeping it forever.

```python
import json
STREAM_IN, STREAM_OUT, CHECKPOINT = "data/shop/stream_in", "data/shop/stream_out", "data/shop/stream_checkpoint"
for folder in [STREAM_IN, STREAM_OUT, CHECKPOINT]:
    shutil.rmtree(folder, ignore_errors=True)
os.makedirs(STREAM_IN)

def new_events(batch, times):
    """Drops one JSON file of events (their times on 2024-03-01) into the input folder."""
    with open(f"{STREAM_IN}/{batch}.json", "w") as f:
        for t in times:
            f.write(json.dumps({"event_time": f"2024-03-01 {t}:00"}) + "\n")

def run_stream():
    """Processes the new files, then prints every window written so far."""
    (spark.readStream.schema("event_time TIMESTAMP").json(STREAM_IN)
          .withWatermark("event_time", "10 minutes")                   # events may be up to 10 minutes late
          .groupBy(F.window("event_time", "5 minutes")).count()       # count per 5-minute window
          .writeStream.outputMode("append").format("parquet")
          .option("path", STREAM_OUT).option("checkpointLocation", CHECKPOINT)
          .trigger(availableNow=True).start().awaitTermination())
    (spark.read.parquet(STREAM_OUT)
          .select(F.date_format("window.start", "HH:mm").alias("window_start"), "count")
          .orderBy("window_start").show())

new_events("batch1", ["10:00", "10:02", "10:07"]); run_stream()
new_events("batch2", ["10:25"]);                   run_stream()
new_events("batch3", ["10:03", "10:31"]);          run_stream()
```

Result:

```
+------------+-----+
|window_start|count|
+------------+-----+
+------------+-----+

+------------+-----+
|window_start|count|
+------------+-----+
|       10:00|    2|
|       10:05|    1|
+------------+-----+

+------------+-----+
|window_start|count|
+------------+-----+
|       10:00|    2|
|       10:05|    1|
+------------+-----+
```

</details>

<details>
<summary><b>B1</b> &nbsp;Clean the names: no spaces at the ends, one space between words, each word capitalized (Aziz Karimov). Show ...</summary>

```python
(signups_raw.select("signup_id",
                     F.initcap(F.regexp_replace(F.trim("full_name"), r"\s+", " ")).alias("name"))
            .show())
```

</details>

<details>
<summary><b>B2</b> &nbsp;Clean the emails: no spaces at the ends and lower case; an empty email becomes null. Show signup_id and email.</summary>

```python
(signups_raw.select("signup_id",
                     F.when(F.trim("email") == "", None).otherwise(F.lower(F.trim("email"))).alias("email"))
            .show(truncate=False))
```

</details>

<details>
<summary><b>B3</b> &nbsp;Flag the valid emails (after cleaning as in B2): text, one @, text, a dot, and letters at the end. A missing ...</summary>

```python
clean_email = F.when(F.trim("email") == "", None).otherwise(F.lower(F.trim("email")))
(signups_raw.select("signup_id", clean_email.alias("email"))
            .withColumn("valid", F.coalesce(F.col("email").rlike(r"^[^@\s]+@[^@\s]+\.[a-z]+$"), F.lit(False)))
            .show(truncate=False))
```

</details>

<details>
<summary><b>B4</b> &nbsp;Keep only the digits of each phone number (digits), and its last 9 digits (phone9). Show signup_id, digits ...</summary>

```python
(signups_raw.withColumn("digits", F.regexp_replace("phone", "[^0-9]", ""))
            .select("signup_id", "digits", F.substring("digits", -9, 9).alias("phone9"))
            .show())
```

</details>

<details>
<summary><b>B5</b> &nbsp;The sign-up dates come in three formats (2024-01-05, 20/01/2024, 2024/02/02). Turn them into one date column ...</summary>

```python
(signups_raw.select("signup_id", "signup_date",
                     F.coalesce(F.try_to_date("signup_date", "yyyy-MM-dd"),
                                F.try_to_date("signup_date", "dd/MM/yyyy"),
                                F.try_to_date("signup_date", "yyyy/MM/dd")).alias("signup_day"))
            .show())
```

</details>

<details>
<summary><b>B6</b> &nbsp;Turn amount (text) into a number amount_value; text that is not a number becomes null. Show signup_id, amount ...</summary>

```python
signups_raw.select("signup_id", "amount", F.col("amount").try_cast("double").alias("amount_value")).show()
```

</details>

<details>
<summary><b>B7</b> &nbsp;Read newsletter and lang from the JSON text in prefs (missing values become null). Show signup_id, newsletter ...</summary>

```python
parsed = F.from_json("prefs", "newsletter BOOLEAN, lang STRING")
signups_raw.select("signup_id", parsed["newsletter"].alias("newsletter"), parsed["lang"].alias("lang")).show()
```

</details>

<details>
<summary><b>B8</b> &nbsp;The same person signed up twice with differently written emails. Clean the emails (as in B2) and keep only ...</summary>

```python
clean = signups_raw.withColumn("email", F.when(F.trim("email") == "", None).otherwise(F.lower(F.trim("email"))))
w = Window.partitionBy("email").orderBy(F.col("signup_id").desc())
(clean.withColumn("rn", F.row_number().over(w))
      .filter((F.col("rn") == 1) | F.col("email").isNull())
      .select("signup_id", "email")
      .orderBy("signup_id")
      .show(truncate=False))
```

</details>

<details>
<summary><b>B9</b> &nbsp;For each customer, collect the quantities of their orders into a sorted array, then keep only the quantities ...</summary>

```python
(orders.filter(F.col("customer_id").isNotNull())
       .groupBy("customer_id")
       .agg(F.array_sort(F.collect_list("quantity")).alias("quantities"))
       .withColumn("above_1", F.filter("quantities", lambda q: q > 1))
       .withColumn("total_above_1", F.aggregate("above_1", F.lit(0), lambda total, q: total + q))
       .orderBy("customer_id")
       .show())
```

</details>

<details>
<summary><b>B10</b> &nbsp;For each category, collect the product prices into a sorted array, and add with_vat: every price × 1.12, ...</summary>

```python
(products.groupBy("category")
         .agg(F.array_sort(F.collect_list("price")).alias("prices"))
         .withColumn("with_vat", F.transform("prices", lambda p: F.round(p * 1.12, 2)))
         .orderBy("category")
         .show(truncate=False))
```

</details>